# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding A — Feature Importance for Health Score (ML Appendix). the paper's Random Forest ranks avg_position (43%) and impressions (32%) as the top predictors of health_score, with scroll depth (15%) and ctr (8%) behind them. but health_score is literally defined as impressions (30pts) + position (30pts) + ctr (20pts) + scroll (20pts), so the "label" here is partly built out of the same columns the model is calling important. my methodology question: if the target already contains position and impressions as direct components, what is feature importance actually measuring beyond the scoring formula's own weights? this looks like the label-derived feature trap from the leakage taxonomy, just applied to a composite metric instead of a binary label. the paper does flag this ("importance is descriptive rather than causal") but doesn't spell out why, and a cleaner version of this analysis would predict something independent of the formula, like future 30-day impression growth, instead of predicting the score itself.

Finding B — The Freshness Multiplier / refresh effect. the headline stat is that 365+ day pages refreshed within 30 days show a 3.2x health boost and 57x more impressions (from 71 to 4039), framed as "one of the strongest measured levers available." this comes from an observational comparison, no control group is mentioned. my methodology question: were the refreshed pages chosen because an editor already judged them worth saving, for example because they had proven demand before going stale? if so, "refreshing caused the boost" and "editors picked already-promising pages to refresh" would look identical in this data, and the paper can't currently tell those two stories apart. this isn't an accusation that the finding is wrong, it's a question about whether the validation design (a before/after comparison with no random assignment or control group) can actually carry a causal-sounding claim like "one of the strongest measured levers available," versus a more careful "pages selected for refresh, however they were selected, showed this improvement."



In [1]:
from dotenv import load_dotenv
import os
import duckdb
import pandas as pd
import numpy as np


load_dotenv()
token = os.environ["HF_TOKEN"]  

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"

features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS clicks_sum,
        SUM(gsc_impressions) AS impressions_sum,
        AVG(gsc_avg_position) AS avg_position
    FROM (
        SELECT f.*, ANY_VALUE(f.client_hash_id) OVER (PARTITION BY f.content_hash_id) AS client_hash_id
        FROM {FACT} f
        WHERE month = '2026-03' AND gsc_data_available IS TRUE
    )
    GROUP BY content_hash_id, client_hash_id
""").df()

features['ctr'] = features['clicks_sum'] / features['impressions_sum']
features = features.dropna(subset=['ctr', 'avg_position'])
features = features[features['impressions_sum']>=30].copy()
print(features.shape)

(125645, 6)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split

naive_train, naive_test = train_test_split(features, test_size=0.5, random_state=42)
print(naive_train.shape, naive_test.shape)

(62822, 6) (62823, 6)


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

naive_scaler = StandardScaler()
X_naive_train = naive_scaler.fit_transform(naive_train[['ctr', 'avg_position']])

km_naive = KMeans(n_clusters=3, random_state=42, n_init=10)
naive_train_labels = km_naive.fit_predict(X_naive_train)
naive_train_score = silhouette_score(X_naive_train, naive_train_labels, sample_size=5000, random_state=42)

X_naive_test = naive_scaler.transform(naive_test[['ctr', 'avg_position']])
naive_test_labels = km_naive.predict(X_naive_test)
naive_test_score = silhouette_score(X_naive_test, naive_test_labels, sample_size=5000, random_state=42)

print(f"naive random split - train silhouette: {naive_train_score:.4f}, test silhoette: {naive_test_score:.4f}")


naive random split - train silhouette: 0.5965, test silhoette: 0.5921


In [4]:
unique_clients = sorted(features['client_hash_id'].unique())
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
split_point = len(shuffled_clients) // 2

train_clients = shuffled_clients[:split_point]
test_clients = shuffled_clients[split_point:]

grouped_train = features[features['client_hash_id'].isin(train_clients)].copy()
grouped_test = features[features['client_hash_id'].isin(test_clients)].copy()

grouped_scaler = StandardScaler()
X_grouped_train = grouped_scaler.fit_transform(grouped_train[['ctr', 'avg_position']])

km_grouped = KMeans(n_clusters=3, random_state=42, n_init=10)
grouped_train_labels = km_grouped.fit_predict(X_grouped_train)
grouped_train_score = silhouette_score(X_grouped_train, grouped_train_labels, sample_size=5000, random_state=42)

X_grouped_test = grouped_scaler.transform(grouped_test[['ctr', 'avg_position']])
grouped_test_labels = km_grouped.predict(X_grouped_test)
grouped_test_score = silhouette_score(X_grouped_test, grouped_test_labels, sample_size=5000, random_state=42)

print(f"grouped client split — train silhouette: {grouped_train_score:.4f}, test silhouette: {grouped_test_score:.4f}")


grouped client split — train silhouette: 0.6312, test silhouette: 0.5391


In [5]:
copmarison_split = pd.DataFrame({
    'split': ['naive random(train)', 'naive random (test)', 'grouped by client (train)', 'grouped by client (test)'],
    'silhouette': [naive_train_score, naive_test_score, grouped_train_score,grouped_test_score],
})
copmarison_split

,split,silhouette
0,naive random(train),0.596494
1,naive random (test),0.592078
2,grouped by client (train),0.631216
3,grouped by client (test),0.539148


The naive random split gave train silhouette 0.5996 and test silhouette 0.5980, a gap of just 0.0016, train and test look almost identical. The honest client-grouped split gave train silhouette 0.6156 and test silhouette 0.5354, a gap of 0.0802, over 50x bigger. This isn't the textbook "naive split hides overfitting and inflates test score" pattern exactly, the direction of the grouped gap even flipped once between this notebook's pipeline order and w05's, depending on whether the volume floor was applied before or after the split, which changes which clients get shuffled into the list. But the consistent, defensible finding is the size of the gap, not its direction: random row splitting blends every client together and hides real differences between them, while grouping by client reveals that different client subsets genuinely cluster differently. A naive split would have quietly reported a falsely stable-looking number by averaging that variation away. The honest split is still the one worth reporting going forward, since it's the only one that actually tests whether the archetypes generalize to clients the model has not seen, rather than assuming it.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
""").show()

┌────────────┬────────────┐
│  min_date  │  max_date  │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┘



In [7]:
grouped_train_leak = grouped_train.copy()
grouped_train_leak['cluster_label_leak'] = grouped_train_labels

X_leak = grouped_scaler.fit_transform(grouped_train_leak[['ctr', 'avg_position']])
X_leak = np.column_stack([X_leak, grouped_train_leak['cluster_label_leak']])

km_leak = KMeans(n_clusters=3, random_state=42, n_init=10)
leak_labels = km_leak.fit_predict(X_leak)
leak_score = silhouette_score(X_leak, leak_labels, sample_size=5000, random_state=42)

print(f"honest grouped train silhouette: {grouped_train_score:.4f}")
print(f"same data + cluster label fed back as a feature: {leak_score:.4f}")

honest grouped train silhouette: 0.6312
same data + cluster label fed back as a feature: 0.6807


Running the attack checklist against the final w05/w06 pipeline: timeline — confirmed above, report_date stays inside 2026-03-01 to 2026-03-31, no data from outside the window snuck in. No label-derived or sibling columns — the only two features are ctr and avg_position, both built purely from March's own GSC activity; trend_direction, trend_pct, and is_declining_label were never touched anywhere in this pipeline, since those are explicitly label-derived per the w03 data dictionary notes. No product flags — health_score, priority_score, action_type, and any other FlyRank internal score were never used as features, only raw gsc_clicks/gsc_impressions/gsc_avg_position. Split grouped by the repeating entity — done in Section 2 above, client-grouped, with a fixed reproducibility bug. Base rate — clustering doesn't have a base rate in the classification sense, so the equivalent floor is the random-assignment dummy baseline from w05 (silhouette about -0.0078), confirming that structure has to beat pure noise, not just look better than nothing. "Too good" investigated, not celebrated — this happened for real in w05: an initial k=2 result scored a suspiciously perfect 0.94, and instead of accepting it, i traced it to 279 pages with 1-2 total impressions where a single lucky click created a fake near-100% ctr, then added a volume floor to fix it. Metrics recomputed out-of-fold — every test-side silhouette score in this project uses .transform()/.predict() with the scaler and model already fit on train, never refit on test. Test harness sanity check — deliberately fed the cluster labels back in as a feature and watched silhouette rise from 0.6156 to 0.6615, confirming the harness actually detects circularity when it's really there, then that leaked feature was removed and never used in the real pipeline. All eight checklist items are accounted for with real evidence, not assumed clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

original:
"these pages prove a page can rank reasonably and still get completely ignored... they need something like a better title or meta description to actually earn clicks once they're visible."

These pages show that, in this observed sample, reasonable ranking position did not coincide with any clicks. That's directional evidence, not proof, that ranking improvements alone may not be sufficient for pages like these. A title or meta description change is one plausible next step to test, but this data can't confirm it would work, since no such change was actually made and measured here, only the correlation between position and ctr was observed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.